# 06 SpaceX Falcon 9 - Interactive Map(Folium)

This notebook is **Step 06** in a multi-step end-to-end data science project that analyzes and predicts **first-stage landing success** for SpaceX Falcon 9 launches.

In this step, we build an interactive **Folium** map to:

- visualize launch site locations
- overlay individual launches colored by landing outcome (success vs. failure)
- compute simple proximity distances (coastline / railway / highway / nearest city) for one representative launch site

**Pipeline overview:**

- **Step 01:** Collect launch data from the SpaceX REST API and create an initial modeling dataset.
- **Step 02:** Scrape a *fixed Wikipedia revision* of Falcon 9 & Falcon Heavy launch tables and export a clean CSV for supplementary analysis.
- **Step 03:** Clean and engineer features, create the landing success label (`Class`) and export the modeling dataset.
- **Step 04:** Load the labeled dataset into SQLite and explore patterns with SQL.
- **Step 05:** Visual EDA + one-hot encoding for modeling.
- **Step 06 (this notebook):** Interactive map + proximity distances.
- **Next steps:** Dashboarding (Dash) and machine learning modeling.

**Input:** `../data/processed/03_dataset_part_2.csv`(from step 03)

**Note on GitHub rendering:** GitHub does not render Folium maps inline inside notebooks. This notebook exports the final map as HTML to:
**Output:** `../data/processed/06_launch_site_map.html` (interactive map export)

## Notebook sections
1. **Setup**
2. **Load the processed data**
3. **Derive unique launch sites**
4. **Build the base map**
5. **Mark all launch sites**
6. **Mark success vs. failure outcomes**
7. **Map helper: live mouse coordinates**
8. **Distance utility (Haversine)**
9. **Proximity example for one launch site**
10. **Save artifacts**
11. **Assumptions**


---


## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd
import folium

# Paths
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

DOCS_DIR = Path('../docs')
DOCS_DIR.mkdir(parents = True, exist_ok = True)

INPUT_CSV = PROCESSED_DIR / '03_dataset_part_2.csv'

OUTPUT_MAP_HTML = PROCESSED_DIR / '06_launch_site_map.html'
OUTPUT_MAP_HTML_DOCS = DOCS_DIR / '06_launch_site_map.html' # GitHub Pages artifact

# Folium plugins / features used later
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

## 2. Load the processed dataset

Step 03 produced a cleaned dataset that already includes launch site names and coordinates.

In [2]:
df = pd.read_csv(INPUT_CSV)

spacex_df = df[['LaunchSite', 'Latitude', 'Longitude', 'Class']].copy()

spacex_df.head()

,LaunchSite,Latitude,Longitude,Class
0,CCSFS SLC 40,28.561857,-80.577366,0
1,CCSFS SLC 40,28.561857,-80.577366,0
2,CCSFS SLC 40,28.561857,-80.577366,0
3,VAFB SLC 4E,34.632093,-120.610829,0
4,CCSFS SLC 40,28.561857,-80.577366,0


## 3. Derive unique launch sites

We create a small `launch_sites_df` with one row per launch site, using the first observed coordinate.

In [3]:
# One row per launch site (first observed coordinate per site)
launch_sites_df = spacex_df.groupby(['LaunchSite'], as_index = False).first()
launch_sites_df = launch_sites_df[['LaunchSite', 'Latitude', 'Longitude']]
launch_sites_df

,LaunchSite,Latitude,Longitude
0,CCSFS SLC 40,28.561857,-80.577366
1,KSC LC 39A,28.608058,-80.603956
2,VAFB SLC 4E,34.632093,-120.610829


## 4. Base map (reference point)

We start from NASA Johnson Space Center for a familiar reference point, then zoom out to include all SpaceX launch sites.

In [4]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]

# Zoom out so all SpaceX sites are visible on map
site_map = folium.Map(location = nasa_coordinate, zoom_start = 5)

In [5]:
# NASA marker (reference point
folium.Circle(
    nasa_coordinate,
    radius = 1000,
    color = '#d35400',
    fill = True,
    fill_opacity = 0.2
).add_child(folium.Popup('NASA Johnson Space Center')).add_to(site_map)

folium.map.Marker(
    nasa_coordinate,
    icon = DivIcon(
        icon_size = (20, 20),
        icon_anchor = (0, 0),
        html = '<div style="font-size: 12; color:#d35400;"><b>NASA JSC</b></div>'
    ),
).add_to(site_map)

## 5. Mark all launch sites

Each launch site is added as:

- a circle with a popup label, and

- a text label (DivIcon) for readability at different zoom levels.

In [6]:
# Add launch site circles + labels
for _, site in launch_sites_df.iterrows():
    site_coordinate = [site['Latitude'], site['Longitude']]
    site_name = site['LaunchSite']

    # Circle + popup
    folium.Circle(
        location = site_coordinate,
        radius = 1000,
        color = '#d35400',
        fill = True,
        fill_color = '#d35400',
        fill_opacity = 0.2,
    ).add_child(folium.Popup(site_name)).add_to(site_map)

    # Text label
    folium.map.Marker(
        site_coordinate,
        icon = DivIcon(
            icon_size = (250, 20),
            icon_anchor = (0, 0),
            html = f'<div style="font-size: 12px; color:#d35400;"><b>{site_name}</b></div>',
        ),
    ).add_to(site_map)

## 6. Mark success vs. failure outcomes

We overlay individual launches using a MarkerCluster. Successful landings are **green** and failures are **red**.

In [7]:
# Quick outcome summary (for context)
outcome_counts = spacex_df['Class'].value_counts().rename({1: 'Success', 0: 'Failure'})
outcome_counts

Class
Success    60
Failure    30
Name: count, dtype: int64

In [8]:
# Cluster individual launches to keep the map readable
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster);

In [9]:
# Assign a color per launch outcome
spacex_df['MarkerColor'] = spacex_df['Class'].apply(lambda x: 'green' if x == 1 else 'red')

In [10]:
# Add one marker per launch (green = success, red = failure)
for _, record in spacex_df.iterrows():
    coordinate = [record['Latitude'], record['Longitude']]

    outcome = 'Success' if record['Class'] == 1 else 'Failure'
    popup_text = f"{record['LaunchSite']} - {outcome}"

    folium.Marker(
        location = coordinate,
        icon = folium.Icon(
            color = record['MarkerColor'],
            icon = 'info-sign',
        ),
        popup = folium.Popup(popup_text, max_width = 300),
    ).add_to(marker_cluster)

## 7. Map helper: live mouse coordinates

The MousePosition plugin is convenient when you want to pick coordinates directly from the map (e.g., coastline points).

In [11]:
# Add mouse position to get coordinates (Latitude/Longitude) on hover
formatter = 'function(num) {return L.Util.formatNum(num, 5);};'
mouse_position = MousePosition(
    position = 'topright',
    separator = ' Long: ',
    empty_string = 'NaN',
    lng_first = False,
    num_digits = 20,
    prefix = 'Lat: ',
    lat_formatter = formatter,
    lng_formatter = formatter,
)

site_map.add_child(mouse_position);

## 8. Distance utility (Haversine)

We use a simple great-circle distance approximation (Haversine) to compute distances between two coordinates.

In [12]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # Approximate radius of Earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

def dist_km(p1, p2):
    return calculate_distance(p1[0], p1[1], p2[0], p2[1])

## 9. Proximity example for one launch site

To keep analysis reproducible, the coordinates below are **recorded** examples near CCAFS LC-40:

- coastline point (as in the original lab),

- a nearby rail facility point from NASA/KSC documentation,

- a major highway connection (SR 401 / Port Canaveral area),

- nearby city coordinates (Cape Canaveral).

Distances are computed in kilometers using `calculate_distance` function.

In [13]:
# Select a launch site to analyze (representative example)
target = 'SLC 40'
launch_site_row = launch_sites_df[
    launch_sites_df['LaunchSite'].str.contains(target, case = False, na = False)
]
launch_site = launch_site_row['LaunchSite'].iloc[0]  # actual name
launch_site_lat = float(launch_site_row['Latitude'].iloc[0])
launch_site_long = float(launch_site_row['Longitude'].iloc[0])
launch_site_coordinate = [launch_site_lat, launch_site_long]

# Recorded example coordinates (picked using MousePosition on the map)
coastline_coordinate = [28.56180, -80.56769]
railway_coordinate = [28.56180, -80.58725]
highway_coordinate = [28.56180, -80.57055]
city_coordinate = [28.38800, -80.60567]

# Compute distances (km)
distance_coastline_km = dist_km(launch_site_coordinate, coastline_coordinate)
distance_railway_km = dist_km(launch_site_coordinate, railway_coordinate)
distance_highway_km = dist_km(launch_site_coordinate, highway_coordinate)
distance_city_km = dist_km(launch_site_coordinate, city_coordinate)

pd.DataFrame(
    {
        'feature': ['coastline', 'railway', 'highway', 'city'],
        'latitude': [
            coastline_coordinate[0],
            railway_coordinate[0],
            highway_coordinate[0],
            city_coordinate[0],
        ],
        'longitude': [
            coastline_coordinate[1],
            railway_coordinate[1],
            highway_coordinate[1],
            city_coordinate[1],
        ],
        'distance_km': [
            distance_coastline_km,
            distance_railway_km,
            distance_highway_km,
            distance_city_km,
        ],
    }
).sort_values('distance_km')

,feature,latitude,longitude,distance_km
2,highway,28.5618,-80.57055,0.665908
0,coastline,28.5618,-80.56769,0.945302
1,railway,28.5618,-80.58725,0.965622
3,city,28.3880,-80.60567,19.535107


In [14]:
# Add proximity markers + distance labels + connecting lines

proximity_features = {
    'Coastline': (coastline_coordinate, distance_coastline_km),
    'Railway': (railway_coordinate, distance_railway_km),
    'Highway': (highway_coordinate, distance_highway_km),
    'City': (city_coordinate, distance_city_km),
}

def add_proximity_feature(feature_name: str, coordinate: list, distance_km: float):
    # Marker
    folium.Marker(
        coordinate,
        icon = folium.Icon(color = 'blue', icon = 'info-sign'),
        popup = folium.Popup(f"{feature_name.title()} (≈ {distance_km:.2f} km)", max_width = 250),
    ).add_to(site_map)

    # Distance label
    label_html = (
        '<div style="'
        'font-size: 11px; '
        'color:#2c3e50; '
        'background-color: rgba(255,255,255,0.85); '
        'padding: 2px 6px; '
        'border-radius: 4px; '
        'border: 1px solid rgba(44,62,80,0.2); '
        '#><b>'
        + f"{feature_name}: {distance_km:.2f} km"
        + '</b></div>'
    )

    folium.map.Marker(
        coordinate,
        icon = DivIcon(
            icon_size = (250, 20),
            icon_anchor = (0, 0),
            html = label_html,
        ),
    ).add_to(site_map)

    # Connecting line
    folium.PolyLine(
        [launch_site_coordinate, coordinate],
        weight = 2,
        opacity = 0.6,
    ).add_to(site_map)

for feature_name, (coord, dist) in proximity_features.items():
    add_proximity_feature(feature_name, coord, dist)

In [15]:
# Optional: display the map in notebook
site_map;

## 10. Save artifacts

Export the interactive map as an HTML artifact.

In [16]:
# Save the final map as HTML
site_map.save(str(OUTPUT_MAP_HTML))
site_map.save(str(OUTPUT_MAP_HTML_DOCS))

## 11. Assumptions

- Launch site coordinates come from the processed data produced in Step 03.
- Proximity points (coastline / railway / highway / city) are **recorded example coordinates** selected using the map mouse-position helper. They are used to demonstrate the distance-calculation workflow, not to create a complete geospatial dataset.
- Distances are computed using a simplified Haversine approximation (great-circle distance). This is sufficient for illustrative analysis at this scale.
- The exported HTML file is the primary artifact for viewing the interactive map (GitHub does not render Folium maps inline in notebooks).

---